In [ ]:
# ==============================================================================
# 1. SETUP & KAGGLE DATASET DOWNLOAD
# ==============================================================================
!pip install -q kaggle

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Ensure Kaggle API token is properly placed
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("⚠️ PLEASE UPLOAD YOUR 'kaggle.json' FILE TO THE COLAB DIRECTORY!")
    # Move the uploaded file to the required hidden directory
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download and unzip the dataset directly from Kaggle
print("📦 Downloading Dataset from Kaggle...")
!kaggle datasets download -d samitha96/coconutdiseases --unzip

# Define the path to the extracted data
DATA_DIR = "/content/Coconut Diseases and Pest Infestations"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/content/" # Fallback

# ==============================================================================
# 2. STRICT DATA SPLITTING (Fixes the 100% Accuracy / Data Leakage)
# ==============================================================================
print("\n📂 Loading datasets and creating strict splits...")

IMG_SIZE = (224, 224) # EfficientNetB0 optimal size
BATCH_SIZE = 32
SEED = 42 # Strict seed prevents data leakage between train and val!

# Load Training Data (80%)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Load Validation Data (20%)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print(f"✅ Classes Detected ({NUM_CLASSES}): {class_names}")

# Optimize dataset performance by caching in memory
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# ==============================================================================
# 3. DATA AUGMENTATION (Further prevents overfitting)
# ==============================================================================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
], name="data_augmentation")

# ==============================================================================
# 4. BUILD THE NEW MODEL: EfficientNetB0
# ==============================================================================
print("\n🏗️ Building EfficientNetB0 model...")

def build_model():
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )

    # Freeze the base model
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x) # Dropout to prevent memorization

    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

model = build_model()
model.summary()

# ==============================================================================
# 5. TRAINING WITH EARLY STOPPING
# ==============================================================================
print("\n🚀 Starting training...")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
]

EPOCHS = 25

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

# ==============================================================================
# 6. EVALUATION AND VISUALIZATIONS (CORRECTED CLASS MAPPING)
# ==============================================================================
print("\n📊 Generating evaluations and visualizations...")

# Extract true labels and model predictions
y_true = []
y_pred_probs = []

for images, labels in val_dataset:
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds)

y_pred = np.argmax(y_pred_probs, axis=1)

# Fix for ValueError: Ensure the report maps labels correctly even if a class is missing in val_dataset
labels_index = np.arange(NUM_CLASSES)

# Classification Report
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, labels=labels_index, target_names=class_names, zero_division=0))

# Plot Accuracy & Loss
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy Plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss Plot
axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Validation Loss', marker='o')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=labels_index)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[2])
axes[2].set_title('Confusion Matrix')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# ==============================================================================
# 7. SAVE THE MODEL
# ==============================================================================
model_save_path = '/content/coconut_disease_efficientnet.keras'
model.save(model_save_path)
print(f"\n✅ Training complete! Model successfully saved as '{model_save_path}'")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from google.colab import files
import matplotlib.pyplot as plt

# 1. Upload an image
print("🥥 Please upload an image of a coconut leaf to test:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 2. Load and preprocess the image
    img_path = '/content/' + filename
    img = image.load_img(img_path, target_size=(224, 224)) # Must match IMG_SIZE
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Create a batch of 1

    # 3. Make the prediction
    predictions = model.predict(img_array)
    predicted_class_index = np.argmax(predictions[0])
    confidence = np.max(predictions[0]) * 100

    # Get the class name
    predicted_class_name = class_names[predicted_class_index]

    # 4. Display the image and the result
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Prediction: {predicted_class_name}\nConfidence: {confidence:.2f}%",
              fontsize=14, fontweight='bold', color='green' if confidence > 70 else 'red')
    plt.show()

    print(f"✅ The AI predicts this is: **{predicted_class_name}** with {confidence:.2f}% confidence.")